# 📚 SQL Ch.1 — SELECT Basics
> BigQuery SQL Reference Guide, Chapter 1: SELECT/FROM · WHERE · AND/OR/NOT · AS · ORDER BY · LIMIT/OFFSET · DISTINCT · Arithmetic
> BigQuery SQL 완전 참조 가이드 1장: SELECT/FROM · WHERE · AND/OR/NOT · AS · ORDER BY · LIMIT/OFFSET · DISTINCT · 산술 연산

---
# 🎯 Learning Objective
Today I want to learn: / 오늘 배우고 싶은 것:
- [x] Write a query that picks specific columns, filters rows with `WHERE`, and sorts with `ORDER BY`  
`WHERE`로 행을 거르고 `ORDER BY`로 정렬해서 원하는 열만 뽑는 쿼리를 작성한다
- [x] Explain why SQL clauses run in a different order than they're written (`FROM` → `WHERE` → `SELECT` → `ORDER BY`)  
SQL 절이 작성 순서와 다르게(`FROM` → `WHERE` → `SELECT` → `ORDER BY`) 실행되는 이유를 설명한다
- [x] Combine `WHERE` + `ORDER BY` + `LIMIT` into the single most common BA query pattern  
`WHERE` + `ORDER BY` + `LIMIT`를 조합해 BA가 가장 자주 쓰는 쿼리 패턴을 완성한다

---
# 🧠 Concept

## What is it?
*(Explain it in your own words.)*

**EN:** A SQL `SELECT` statement is how you ask a database for data. `SELECT` names the columns you want, `FROM` names the table, and clauses like `WHERE`, `ORDER BY`, and `LIMIT` narrow down and arrange the rows you get back. This chapter covers the building blocks — SELECT/FROM, WHERE, AND/OR/NOT, AS, ORDER BY, LIMIT/OFFSET, DISTINCT, and arithmetic — that every later SQL topic (JOIN, GROUP BY, window functions) is built on top of.

**KR:** SQL `SELECT` 문은 데이터베이스에 데이터를 요청하는 방법입니다. `SELECT`는 원하는 열의 이름을, `FROM`은 테이블의 이름을 지정하고, `WHERE`·`ORDER BY`·`LIMIT` 같은 절이 돌려받는 행을 좁히고 정렬합니다. 이번 챕터는 SELECT/FROM, WHERE, AND/OR/NOT, AS, ORDER BY, LIMIT/OFFSET, DISTINCT, 산술 연산이라는 기본 구성 요소를 다루며, 이후 배울 모든 SQL 주제(JOIN, GROUP BY, 윈도우 함수)가 이 위에 세워집니다.

## Why do we use it?
*(When is it useful?)*

**EN:** Every analysis starts by pulling the right slice of data — the right columns, the right rows, in the right order. Without `WHERE`/`ORDER BY`/`LIMIT`, you'd have to export entire tables and filter them by hand in a spreadsheet. These clauses let the database do that filtering and sorting for you, on datasets far larger than Excel could ever open.

**KR:** 모든 분석은 데이터의 알맞은 일부 — 알맞은 열, 알맞은 행을, 알맞은 순서로 — 가져오는 것에서 시작합니다. `WHERE`/`ORDER BY`/`LIMIT`이 없다면 테이블 전체를 내보낸 뒤 스프레드시트에서 손으로 걸러야 합니다. 이 절들은 그 필터링과 정렬 작업을 데이터베이스가 대신 처리하게 해주며, 엑셀로는 열어볼 엄두도 못 낼 규모의 데이터에서도 동작합니다.

## When is it used in Business Analytics?
*(Real-world use case)*

**EN:** A BA writes this pattern dozens of times a week: "show me completed orders over ₩50,000, newest first, top 10 only" is literally `SELECT ... FROM ... WHERE ... ORDER BY ... LIMIT 10`. It's the query you run before any deep-dive — to sanity-check a table, spot the biggest customers, or pull a quick list for a Slack message.

**KR:** BA는 이 패턴을 일주일에도 수십 번씩 씁니다. "5만원 넘는 완료 주문을 최신순으로 상위 10건만 보여줘"는 그대로 `SELECT ... FROM ... WHERE ... ORDER BY ... LIMIT 10`이 됩니다. 본격적인 분석에 들어가기 전 테이블을 점검하거나, 가장 큰 고객을 찾거나, 슬랙 메시지에 넣을 간단한 목록을 뽑을 때 가장 먼저 실행하는 쿼리입니다.

**Comparison / 비교표 — same idea, three tools:**

| Task / 작업 | SQL | Pandas | Excel |
|---|---|---|---|
| Pick columns / 열 선택 | `SELECT col1, col2` | `df[["col1","col2"]]` | Hide/delete columns |
| Filter rows / 행 필터링 | `WHERE price > 100` | `df[df["price"]>100]` | AutoFilter |
| Sort / 정렬 | `ORDER BY amount DESC` | `df.sort_values("amount", ascending=False)` | Sort Z→A |
| Top N rows / 상위 N개 | `LIMIT 5` | `df.head(5)` | (manual) |
| Unique values / 고유값 | `DISTINCT col` | `df["col"].unique()` | Remove Duplicates |

---
# 📝 Syntax

## Basic Syntax
The full shape of a basic query: pick columns, pick a table, optionally filter/sort/limit.
기본 쿼리의 전체 골격: 열을 고르고, 테이블을 고르고, 필요하면 필터/정렬/제한을 추가합니다.

In [1]:
# --- Environment setup / 환경 설정 ---
# We use DuckDB: a free, in-memory SQL engine that understands BigQuery-style syntax
# almost 1:1 (window functions, QUALIFY, ROLLUP, STRING_AGG, etc.), and can query
# pandas DataFrames directly by name -- no separate "load data" step needed.
# DuckDB는 무료 인메모리 SQL 엔진으로, BigQuery 문법(윈도우 함수, QUALIFY, ROLLUP,
# STRING_AGG 등)을 거의 그대로 이해하고, pandas DataFrame을 이름으로 바로 조회할 수
# 있습니다. 별도의 "데이터 로드" 단계가 필요 없습니다.
import duckdb
import pandas as pd
from IPython.display import display

def run(sql: str) -> pd.DataFrame:
    """Execute a SQL string against DuckDB and return the result as a DataFrame.
    SQL 문자열을 DuckDB에서 실행하고 결과를 DataFrame으로 반환합니다."""
    return duckdb.sql(sql).df()

orders = pd.DataFrame({
    "order_id":    [1001, 1002, 1003, 1004, 1005],
    "customer_id": ["C01", "C02", "C01", "C03", "C02"],
    "order_date":  ["2024-01-15","2024-01-22","2024-02-05","2024-02-18","2024-03-01"],
    "amount":      [45000, 32000, 61000, 28000, 95000],
    "status":      ["완료", "완료", "취소", "완료", "완료"],
})

# The full skeleton of a query -- every clause below is optional except SELECT/FROM.
# 쿼리의 전체 골격 -- SELECT/FROM을 제외한 모든 절은 선택 사항입니다.
sql = """
SELECT order_id, customer_id, amount   -- 1. which columns / 어떤 열
FROM orders                            -- 2. which table   / 어떤 테이블
WHERE status = '완료'                  -- 3. which rows    / 어떤 행 (optional)
ORDER BY amount DESC                   -- 4. what order    / 어떤 순서 (optional)
LIMIT 3                                -- 5. how many      / 몇 개    (optional)
"""
display(run(sql))


,order_id,customer_id,amount
0,1005,C02,95000
1,1001,C01,45000
2,1002,C02,32000


## Common Variations

In [2]:
# SELECT * -- every column, every row
# SELECT * -- 모든 열, 모든 행
display(run("SELECT * FROM orders"))

# Column order in the result follows the order you write in SELECT, not the table's original order
# 결과의 열 순서는 SELECT에 쓴 순서를 따름 -- 원본 테이블 순서와 무관
display(run("SELECT amount, order_id, status FROM orders"))


,order_id,customer_id,order_date,amount,status
0,1001,C01,2024-01-15,45000,완료
1,1002,C02,2024-01-22,32000,완료
2,1003,C01,2024-02-05,61000,취소
3,1004,C03,2024-02-18,28000,완료
4,1005,C02,2024-03-01,95000,완료


,amount,order_id,status
0,45000,1001,완료
1,32000,1002,완료
2,61000,1003,취소
3,28000,1004,완료
4,95000,1005,완료


---
# 🧪 Small Examples

## Example 1 — WHERE: Filtering Rows / 행 필터링
**EN:** `WHERE` keeps only the rows that match a condition. Comparison operators: `=`, `!=` (or `<>`), `>`, `>=`, `<`, `<=`. String values need **single** quotes, and in BigQuery string comparison is case-sensitive and whitespace-sensitive.  
**KR:** `WHERE`는 조건에 맞는 행만 남깁니다. 비교 연산자: `=`, `!=`(또는 `<>`), `>`, `>=`, `<`, `<=`. 문자열 값은 **작은따옴표**를 써야 하고, BigQuery의 문자열 비교는 대소문자와 공백을 구분합니다.

In [3]:
products = pd.DataFrame({
    "product_id": ["P01","P02","P03","P04","P05"],
    "name":       ["노트북","마우스","키보드","셔츠","청바지"],
    "category":   ["전자","전자","전자","의류","의류"],
    "price":      [1200000, 25000, 45000, 35000, 89000],
    "cost":       [900000, 15000, 30000, 20000, 50000],
})

print("-- equality / 같음 --")
display(run("SELECT * FROM products WHERE category = '전자'"))

print("-- greater-or-equal / 이상 --")
display(run("SELECT * FROM products WHERE price >= 45000"))

print("-- not-equal / 같지 않음 (!= or <>) --")
display(run("SELECT * FROM products WHERE category != '의류'"))


-- equality / 같음 --


,product_id,name,category,price,cost
0,P01,노트북,전자,1200000,900000
1,P02,마우스,전자,25000,15000
2,P03,키보드,전자,45000,30000


-- greater-or-equal / 이상 --


,product_id,name,category,price,cost
0,P01,노트북,전자,1200000,900000
1,P03,키보드,전자,45000,30000
2,P05,청바지,의류,89000,50000


-- not-equal / 같지 않음 (!= or <>) --


,product_id,name,category,price,cost
0,P01,노트북,전자,1200000,900000
1,P02,마우스,전자,25000,15000
2,P03,키보드,전자,45000,30000


## Example 2 — AND / OR / NOT: Compound Conditions / 복합 조건
**EN:** Combine multiple conditions with `AND` (both must be true), `OR` (either can be true), and `NOT` (flip a condition). `AND` binds tighter than `OR` — when mixing them, use parentheses `()` to make your intent explicit, or the result can silently differ from what you meant.  
**KR:** `AND`(둘 다 참), `OR`(둘 중 하나만 참), `NOT`(조건 반전)으로 여러 조건을 조합합니다. `AND`가 `OR`보다 우선순위가 높으므로, 섞어 쓸 때는 괄호 `()`로 의도를 명확히 해야 합니다 — 안 그러면 의도와 다른 결과가 조용히 나올 수 있습니다.

In [4]:
employees = pd.DataFrame({
    "emp_id": ["E01","E02","E03","E04","E05","E06"],
    "name":   ["김민수","이영희","박준호","최서연","정대현","윤소영"],
    "dept":   ["영업","마케팅","개발","영업","마케팅","개발"],
    "salary": [4200000, 3800000, 5100000, 3500000, 4500000, 5800000],
})

print("-- AND: dev team AND salary >= 5,000,000 / 개발팀이면서 연봉 5백만 이상 --")
display(run("SELECT name, dept, salary FROM employees WHERE dept='개발' AND salary>=5000000"))

print("-- NOT: everyone except sales / 영업팀이 아닌 직원 --")
display(run("SELECT name, dept FROM employees WHERE NOT dept = '영업'"))

print("-- Intent: (dev AND salary>=5M) OR marketing -- parentheses required")
print("-- 의도: (개발팀이면서 연봉 5백 이상) 또는 (마케팅팀) -- 괄호 필수")
display(run("SELECT name, dept, salary FROM employees WHERE (dept='개발' AND salary>=5000000) OR dept='마케팅'"))


-- AND: dev team AND salary >= 5,000,000 / 개발팀이면서 연봉 5백만 이상 --


,name,dept,salary
0,박준호,개발,5100000
1,윤소영,개발,5800000


-- NOT: everyone except sales / 영업팀이 아닌 직원 --


,name,dept
0,이영희,마케팅
1,박준호,개발
2,정대현,마케팅
3,윤소영,개발


-- Intent: (dev AND salary>=5M) OR marketing -- parentheses required
-- 의도: (개발팀이면서 연봉 5백 이상) 또는 (마케팅팀) -- 괄호 필수


,name,dept,salary
0,이영희,마케팅,3800000
1,박준호,개발,5100000
2,정대현,마케팅,4500000
3,윤소영,개발,5800000


## Example 3 — AS: Aliasing / 별칭
**EN:** `AS` renames a column (or a computed expression, or a table) in the *output* only — it doesn't touch the real column name. The `AS` keyword itself is optional, but keeping it makes queries far more readable.  
**KR:** `AS`는 열(또는 계산식, 테이블)의 이름을 *결과에서만* 바꿉니다 — 실제 열 이름은 그대로입니다. `AS` 키워드는 생략할 수 있지만, 써주는 것이 가독성에 훨씬 좋습니다.

In [5]:
products3 = pd.DataFrame({
    "product_id": ["P01","P02","P03"],
    "name":       ["노트북","마우스","키보드"],
    "price":      [1200000, 25000, 45000],
    "cost":       [900000, 15000, 30000],
})

print("-- rename columns in the output / 결과 열 이름 바꾸기 --")
display(run("SELECT product_id AS id, name AS product_name, price AS 판매가 FROM products3"))

print("-- alias a computed expression / 계산식에 별칭 붙이기 --")
display(run("SELECT name, price - cost AS profit FROM products3"))

print("-- table alias -- shortens long table names, essential once we reach JOIN --")
print("-- 테이블 별칭 -- 긴 테이블명을 줄여줌, JOIN에서 특히 중요해짐 --")
display(run("SELECT p.name, p.price FROM products3 AS p WHERE p.price > 30000"))

# ⚠️ In real BigQuery, an alias CANNOT be reused inside WHERE (SELECT runs after WHERE).
# ⚠️ 실제 BigQuery에서는 별칭을 WHERE 안에서 다시 쓸 수 없습니다 (WHERE가 SELECT보다 먼저 실행됨).
# Note: DuckDB (this notebook's engine) is more lenient and actually allows it --
# but write it the portable way below so the same query also works on BigQuery.
# 참고: 이 노트북이 쓰는 DuckDB는 더 관대해서 실제로는 허용합니다 --
# 하지만 BigQuery에서도 그대로 동작하도록 아래처럼 이식성 있게 작성하세요.
print("-- portable version: repeat the expression instead of the alias --")
print("-- 이식성 있는 버전: 별칭 대신 식을 그대로 반복 --")
display(run("SELECT name, price - cost AS profit FROM products3 WHERE price - cost > 10000"))


-- rename columns in the output / 결과 열 이름 바꾸기 --


,id,product_name,판매가
0,P01,노트북,1200000
1,P02,마우스,25000
2,P03,키보드,45000


-- alias a computed expression / 계산식에 별칭 붙이기 --


,name,profit
0,노트북,300000
1,마우스,10000
2,키보드,15000


-- table alias -- shortens long table names, essential once we reach JOIN --
-- 테이블 별칭 -- 긴 테이블명을 줄여줌, JOIN에서 특히 중요해짐 --


,name,price
0,노트북,1200000
1,키보드,45000


-- portable version: repeat the expression instead of the alias --
-- 이식성 있는 버전: 별칭 대신 식을 그대로 반복 --


,name,profit
0,노트북,300000
1,키보드,15000


## Example 4 — ORDER BY: Sorting / 정렬
**EN:** `ORDER BY` sorts the result. `ASC` (ascending, low→high) is the default so it's usually omitted; `DESC` (descending, high→low) must be written explicitly. List multiple columns to break ties: the first column sorts first, the second breaks ties within it.  
**KR:** `ORDER BY`는 결과를 정렬합니다. `ASC`(오름차순)가 기본값이라 보통 생략하고, `DESC`(내림차순)는 명시해야 합니다. 여러 열을 나열하면 앞의 열로 먼저 정렬하고, 같은 값일 때 다음 열로 다시 정렬합니다.

In [6]:
employees5 = pd.DataFrame({
    "emp_id": ["E01","E02","E03","E04","E05","E06"],
    "name":   ["김민수","이영희","박준호","최서연","정대현","윤소영"],
    "dept":   ["영업","마케팅","개발","영업","마케팅","개발"],
    "salary": [4200000, 3800000, 5100000, 3500000, 4500000, 5800000],
    "years":  [3, 5, 7, 2, 6, 9],
})

print("-- single column, DESC / 단일 열, 내림차순 --")
display(run("SELECT name, dept, salary FROM employees5 ORDER BY salary DESC"))

print("-- multi-column: dept ASC, then salary DESC within each dept --")
print("-- 다중 열: 부서 오름차순, 같은 부서 안에서는 연봉 내림차순 --")
display(run("SELECT name, dept, salary FROM employees5 ORDER BY dept ASC, salary DESC"))

# 💡 ORDER BY CAN use a SELECT alias (unlike WHERE) -- it runs after SELECT.
# 💡 ORDER BY는 (WHERE와 달리) SELECT에서 만든 별칭을 쓸 수 있습니다 -- SELECT 이후에 실행되기 때문.
print("-- ORDER BY reusing a computed alias / 계산된 별칭을 ORDER BY에서 재사용 --")
display(run("SELECT name, salary - 4000000 AS over_avg FROM employees5 ORDER BY over_avg DESC"))


-- single column, DESC / 단일 열, 내림차순 --


,name,dept,salary
0,윤소영,개발,5800000
1,박준호,개발,5100000
2,정대현,마케팅,4500000
3,김민수,영업,4200000
4,이영희,마케팅,3800000
5,최서연,영업,3500000


-- multi-column: dept ASC, then salary DESC within each dept --
-- 다중 열: 부서 오름차순, 같은 부서 안에서는 연봉 내림차순 --


,name,dept,salary
0,윤소영,개발,5800000
1,박준호,개발,5100000
2,정대현,마케팅,4500000
3,이영희,마케팅,3800000
4,김민수,영업,4200000
5,최서연,영업,3500000


-- ORDER BY reusing a computed alias / 계산된 별칭을 ORDER BY에서 재사용 --


,name,over_avg
0,윤소영,1800000
1,박준호,1100000
2,정대현,500000
3,김민수,200000
4,이영희,-200000
5,최서연,-500000


## Example 5 — LIMIT / OFFSET: Limiting Row Count / 행 수 제한
**EN:** `LIMIT n` returns only the first `n` rows. `OFFSET n` skips the first `n` rows before applying the limit — the combination is basically pagination ("page 2 of results"). **Always pair `LIMIT` with `ORDER BY`** — without a defined order, which rows come back is not guaranteed.  
**KR:** `LIMIT n`은 처음 `n`개 행만 반환합니다. `OFFSET n`은 제한을 적용하기 전에 처음 `n`개 행을 건너뛰며, 둘을 합치면 사실상 페이지네이션("결과의 2페이지")입니다. **`LIMIT`은 항상 `ORDER BY`와 함께 쓰세요** — 정렬 기준이 없으면 어떤 행이 돌아올지 보장되지 않습니다.

In [7]:
orders6 = pd.DataFrame({
    "order_id":    [1001, 1002, 1003, 1004, 1005, 1006],
    "customer_id": ["C01","C02","C01","C03","C02","C04"],
    "amount":      [45000, 32000, 61000, 28000, 95000, 15000],
    "order_date":  ["2024-01-15","2024-01-22","2024-02-05","2024-02-18","2024-03-01","2024-03-15"],
})

print("-- top 3 most recent orders / 최근 3건 --")
display(run("SELECT order_id, customer_id, amount FROM orders6 ORDER BY order_date DESC LIMIT 3"))

print("-- paging: rows 3-5 (skip the first 2) / 3~5번째 행 (앞 2개 건너뛰기) --")
display(run("SELECT order_id, amount FROM orders6 ORDER BY order_id LIMIT 3 OFFSET 2"))


-- top 3 most recent orders / 최근 3건 --


,order_id,customer_id,amount
0,1006,C04,15000
1,1005,C02,95000
2,1004,C03,28000


-- paging: rows 3-5 (skip the first 2) / 3~5번째 행 (앞 2개 건너뛰기) --


,order_id,amount
0,1003,61000
1,1004,28000
2,1005,95000


## Example 6 — DISTINCT: Removing Duplicates / 중복 제거
**EN:** `DISTINCT` collapses duplicate rows down to one. With a single column, it lists unique values; with multiple columns, "duplicate" means the *whole combination* repeats. `COUNT(DISTINCT col)` counts how many unique values exist.  
**KR:** `DISTINCT`는 중복된 행을 하나로 합칩니다. 열이 하나면 고유값 목록이 되고, 열이 여러 개면 "중복"은 그 *조합 전체*가 반복되는 경우를 뜻합니다. `COUNT(DISTINCT col)`은 고유값이 몇 개인지 셉니다.

In [8]:
orders7 = pd.DataFrame({
    "order_id":    [1001, 1002, 1003, 1004, 1005, 1006],
    "customer_id": ["C01","C02","C01","C03","C02","C01"],
    "region":      ["서울","부산","서울","인천","부산","서울"],
    "status":      ["완료","완료","취소","완료","완료","완료"],
})

print("-- which customers ordered at all? / 어떤 고객이 주문했는지 --")
display(run("SELECT DISTINCT customer_id FROM orders7"))

print("-- unique (customer, region) combinations / 고객+지역 조합 기준 중복 제거 --")
display(run("SELECT DISTINCT customer_id, region FROM orders7"))

print("-- how many distinct regions? / 지역이 몇 개인지 --")
display(run("SELECT COUNT(DISTINCT region) AS region_count FROM orders7"))


-- which customers ordered at all? / 어떤 고객이 주문했는지 --


,customer_id
0,C01
1,C02
2,C03


-- unique (customer, region) combinations / 고객+지역 조합 기준 중복 제거 --


,customer_id,region
0,C02,부산
1,C01,서울
2,C03,인천


-- how many distinct regions? / 지역이 몇 개인지 --


,region_count
0,3


## Example 7 — Arithmetic Operations: Creating New Columns / 산술 연산으로 새 열 만들기
**EN:** You can compute new columns right inside `SELECT` using `+ - * /`. In BigQuery, `/` between two integers **always** returns a decimal (e.g. `10/3 = 3.333...`) — use `//` for integer floor-division and `MOD()` for the remainder.  
**KR:** `SELECT` 안에서 `+ - * /`로 바로 새 열을 계산할 수 있습니다. BigQuery에서 정수 `/` 정수는 **항상** 소수를 반환합니다(예: `10/3 = 3.333...`) — 정수 나눗셈은 `//`, 나머지는 `MOD()`를 씁니다.

In [9]:
products8 = pd.DataFrame({
    "product_id": ["P01","P02","P03","P04"],
    "name":       ["노트북","마우스","키보드","티셔츠"],
    "price":      [1200000, 25000, 45000, 35000],
    "cost":       [900000, 15000, 30000, 20000],
    "quantity":   [3, 20, 12, 15],
})

print("-- profit, revenue, margin % / 이익, 매출, 마진율(%) --")
sql = """
SELECT
    name,
    price - cost                          AS profit,     -- 이익
    price * quantity                      AS revenue,    -- 매출
    ROUND((price - cost) / price * 100, 1) AS margin_pct  -- 마진율(%)
FROM products8
"""
display(run(sql))

print("-- division vs floor-division vs remainder / 나눗셈 vs 정수 나눗셈 vs 나머지 --")
display(run("SELECT 10/3 AS division, 10//3 AS floor_div, MOD(10,3) AS remainder"))


-- profit, revenue, margin % / 이익, 매출, 마진율(%) --


,name,profit,revenue,margin_pct
0,노트북,300000,3600000,25.0
1,마우스,10000,500000,40.0
2,키보드,15000,540000,33.3
3,티셔츠,15000,525000,42.9


-- division vs floor-division vs remainder / 나눗셈 vs 정수 나눗셈 vs 나머지 --


,division,floor_div,remainder
0,3.333333,3,1


## Example 8 — Common Combinations: WHERE + ORDER BY + LIMIT / 자주 쓰는 조합
**EN:** These clauses are almost always used together. Pattern A ("top N that meet a condition") is the single most common query shape a BA writes. Pattern B shows arithmetic + WHERE together — remember, the computed alias still can't be reused in `WHERE` (Example 3), so the expression is repeated.  
**KR:** 이 절들은 거의 항상 함께 쓰입니다. 패턴 A("조건을 만족하는 상위 N개")는 BA가 가장 많이 쓰는 쿼리 형태입니다. 패턴 B는 산술 연산 + WHERE를 함께 보여주는데, 계산된 별칭은 여전히 `WHERE`에서 재사용할 수 없으므로(예제 3) 식을 그대로 반복합니다.

In [10]:
orders9 = pd.DataFrame({
    "order_id":    [1001, 1002, 1003, 1004, 1005, 1006],
    "customer_id": ["C01","C02","C01","C03","C02","C01"],
    "region":      ["서울","부산","서울","인천","부산","서울"],
    "status":      ["완료","완료","취소","완료","완료","완료"],
    "amount":      [45000, 32000, 61000, 28000, 95000, 15000],
})

print("-- Pattern A: top 3 completed orders by amount / 완료된 주문 중 금액 상위 3건 --")
sql_a = """
SELECT order_id, customer_id, amount
FROM orders9
WHERE status = '완료'
ORDER BY amount DESC
LIMIT 3
"""
display(run(sql_a))

print("-- Pattern B: margin >= 35%, highest margin first / 마진율 35% 이상, 마진율 내림차순 --")
sql_b = """
SELECT name, price, cost, ROUND((price-cost)/price*100, 1) AS margin_pct
FROM products8
WHERE (price - cost) / price * 100 >= 35
ORDER BY margin_pct DESC
"""
display(run(sql_b))


-- Pattern A: top 3 completed orders by amount / 완료된 주문 중 금액 상위 3건 --


,order_id,customer_id,amount
0,1005,C02,95000
1,1001,C01,45000
2,1002,C02,32000


-- Pattern B: margin >= 35%, highest margin first / 마진율 35% 이상, 마진율 내림차순 --


,name,price,cost,margin_pct
0,티셔츠,35000,20000,42.9
1,마우스,25000,15000,40.0


## Example 9 — Practice / 실습 문제
**EN:** Fill in each `________` blank below, then remove the `#` in front of the matching `display(run(...))` line to check your answer. Hints: `=` `>` `DESC` `LIMIT` `ROUND`  
**KR:** 아래 `________` 빈칸을 채운 뒤, 해당하는 `display(run(...))` 줄 앞의 `#`을 지우고 실행해서 답을 확인하세요. 힌트: `=` `>` `DESC` `LIMIT` `ROUND`

In [13]:
employees_p = pd.DataFrame({
    "emp_id": ["E01","E02","E03","E04","E05","E06"],
    "name":   ["김민수","이영희","박준호","최서연","정대현","윤소영"],
    "dept":   ["영업","마케팅","개발","영업","마케팅","개발"],
    "salary": [4200000, 3800000, 5100000, 3500000, 4500000, 5800000],
    "years":  [3, 5, 7, 2, 6, 9],
})

# Q1. Employees who are in dev OR earn over 4,000,000 -- name/dept/salary, salary DESC
# Q1. 개발팀이거나 연봉이 4,000,000 초과인 직원 -- 이름/부서/연봉을 연봉 내림차순으로
q1 = """
SELECT name, dept, salary
FROM employees_p
WHERE dept = '개발'
   OR salary > 4000000
ORDER BY salary DESC
"""
display(run(q1))   # <- uncomment once the blanks are filled / 빈칸을 채운 뒤 주석 해제

# Q2. name, salary, and monthly salary (salary/12, 1 decimal) -- top 3 by monthly salary
# Q2. 이름, 연봉, 월급(연봉/12, 소수 첫째 자리) -- 월급 기준 상위 3명만
q2 = """
SELECT
    name,
    salary,
    ROUND(salary / 12, 1) AS monthly_salary
FROM employees_p
ORDER BY monthly_salary DESC
LIMIT 3
"""
display(run(q2))   # <- uncomment once the blanks are filled / 빈칸을 채운 뒤 주석 해제

print("✏️  Fill in the ________ blanks above, uncomment the display() lines, then re-run this cell.")
print("✏️  위 ________ 빈칸을 채우고 display() 줄의 주석을 해제한 뒤 이 셀을 다시 실행하세요.")


,name,dept,salary
0,윤소영,개발,5800000
1,박준호,개발,5100000
2,정대현,마케팅,4500000
3,김민수,영업,4200000


,name,salary,monthly_salary
0,윤소영,5800000,483333.3
1,박준호,5100000,425000.0
2,정대현,4500000,375000.0


✏️  Fill in the ________ blanks above, uncomment the display() lines, then re-run this cell.
✏️  위 ________ 빈칸을 채우고 display() 줄의 주석을 해제한 뒤 이 셀을 다시 실행하세요.


<details>
<summary>🔑 Answer / 정답 (click to expand / 클릭해서 펼치기)</summary>

```sql
-- Q1
SELECT name, dept, salary
FROM employees_p
WHERE dept = '개발'
   OR salary > 4000000
ORDER BY salary DESC

-- Q2
SELECT
    name,
    salary,
    ROUND(salary / 12, 1) AS monthly_salary
FROM employees_p
ORDER BY monthly_salary DESC
LIMIT 3
```
</details>

---
# ⚠️ Common Mistakes

**Mistake 1 — `SELECT *` on wide/large tables**
- EN: `SELECT *` pulls every column whether you need it or not. BigQuery bills by columns scanned, so `SELECT *` is both slower and more expensive than naming only what you need.
- KR: `SELECT *`는 필요 여부와 상관없이 모든 열을 가져옵니다. BigQuery는 스캔한 열 단위로 과금하므로 `SELECT *`는 필요한 열만 지정하는 것보다 느리고 비용도 더 듭니다.
- ✅ Fix / 해결법: Name the columns you actually need — `SELECT order_id, amount` instead of `SELECT *`.  
실제로 필요한 열만 명시하세요.

**Mistake 2 — Forgetting parentheses when mixing AND / OR**
- EN: `AND` has higher precedence than `OR`. `WHERE dept='개발' AND salary>=5000000 OR dept='마케팅'` is silently read as `WHERE (dept='개발' AND salary>=5000000) OR dept='마케팅'` — which may not be the grouping you intended.
- KR: `AND`는 `OR`보다 우선순위가 높습니다. `WHERE dept='개발' AND salary>=5000000 OR dept='마케팅'`은 조용히 `WHERE (dept='개발' AND salary>=5000000) OR dept='마케팅'`로 해석되는데, 이게 의도한 묶음이 아닐 수 있습니다.
- ✅ Fix / 해결법: Always write parentheses to make grouping explicit, even when the default happens to match your intent.  
의도한 결과와 우연히 같더라도 항상 괄호로 묶음을 명시하세요.

**Mistake 3 — Reusing a SELECT alias inside WHERE**
- EN: SQL's real execution order is `FROM → WHERE → SELECT → ORDER BY`, so `WHERE` runs *before* the alias defined in `SELECT` exists. `WHERE profit > 10000` fails if `profit` was only just aliased in `SELECT`.
- KR: SQL의 실제 실행 순서는 `FROM → WHERE → SELECT → ORDER BY`이므로, `WHERE`는 `SELECT`에서 만든 별칭이 존재하기 *전에* 실행됩니다. `profit`이 `SELECT`에서 방금 만든 별칭이라면 `WHERE profit > 10000`은 실패합니다.
- ✅ Fix / 해결법: Repeat the full expression in `WHERE` (`WHERE price - cost > 10000`), or use a subquery/CTE (Chapter 4).  
`WHERE`에 전체 식을 반복해서 쓰거나(`WHERE price - cost > 10000`), 서브쿼리/CTE를 사용하세요(4장).

**Mistake 4 — `LIMIT` without `ORDER BY`**
- EN: Without an `ORDER BY`, a database is free to return rows in any order — often storage order, which can change between runs. `LIMIT 5` alone gives you *some* 5 rows, not necessarily the top 5 by any meaningful measure.
- KR: `ORDER BY`가 없으면 데이터베이스는 어떤 순서로든 행을 반환할 수 있으며, 이는 실행할 때마다 달라질 수 있습니다. `LIMIT 5`만 쓰면 *어떤* 5개 행이지, 의미 있는 기준의 상위 5개가 아닐 수 있습니다.
- ✅ Fix / 해결법: Always pair `LIMIT` with an `ORDER BY` that defines what "top" means.  
`LIMIT`은 항상 "상위"의 기준이 되는 `ORDER BY`와 함께 쓰세요.

---
# 💡 Tips
Useful tips or shortcuts / 유용한 팁과 단축법

- The `AS` keyword is optional, but keep it — `price 판매가` technically works the same as `price AS 판매가`, but it's much easier to misread.  
`AS` 키워드는 생략 가능하지만 써주는 게 좋습니다 — `price 판매가`도 `price AS 판매가`와 동일하게 동작하지만 오독하기 훨씬 쉽습니다.
- `ORDER BY` defaults to `ASC`, so you only ever need to type `DESC` — one less thing to remember for the common case.  
`ORDER BY`의 기본값은 `ASC`이므로 사실상 `DESC`만 기억하면 됩니다 — 흔한 경우를 위해 외울 게 하나 줄어드는 셈입니다.
- Remember the real run order — `FROM → WHERE → SELECT → ORDER BY` — and half of SQL's "weird rules" (like Mistake 3 above) stop feeling weird.  
실제 실행 순서 `FROM → WHERE → SELECT → ORDER BY`를 기억해 두면 SQL의 "이상한 규칙"(위 Mistake 3 같은) 절반이 더 이상 이상하게 느껴지지 않습니다.
- `LIMIT n OFFSET m` is pagination: think "give me page `m/n + 1`," which is exactly what an app does when you click "next page."  
`LIMIT n OFFSET m`은 페이지네이션입니다: "`m/n + 1`페이지를 달라"는 뜻이며, 앱에서 "다음 페이지"를 누를 때 내부적으로 벌어지는 일이 정확히 이것입니다.

---
# 🔗 Related Concepts

```
SQL Learning Roadmap (this guide) / SQL 학습 로드맵 (이 가이드)
──────────────────────────────────────────────
 1. SELECT Basics              ← ★ YOU ARE HERE / 지금 여기
 2. Aggregation & GROUP BY
 3. JOIN
 4. Subquery & CTE
 5. Conditions & NULL Handling
 6. String & Date Functions
 7. Window Functions
 8. BA-Specific Patterns
```

```
Inside this chapter -- clause build-up / 이 챕터 내부 -- 절이 쌓이는 순서
──────────────────────────────────────────────
   SELECT  →  FROM  →  WHERE  →  ORDER BY  →  LIMIT
  (열 선택)  (테이블)   (필터)     (정렬)      (개수 제한)
```

*How is today's topic connected to other concepts?*

**EN:** This is the very first chapter, so there's no "before" — but everything after it leans on these clauses. Chapter 2 (`GROUP BY`) collapses many rows into fewer using the same `WHERE`/`ORDER BY` you just learned, plus a new clause (`HAVING`) that filters *after* grouping. Chapter 3 (`JOIN`) reuses `AS` constantly to alias tables. And the `WHERE` + `ORDER BY` + `LIMIT` combo from Example 8 reappears, almost unchanged, inside every later chapter's "common combinations" section.

**KR:** 이번이 첫 챕터라 "이전"은 없지만, 이후 모든 내용이 이 절들 위에 세워집니다. 2장(`GROUP BY`)은 방금 배운 `WHERE`/`ORDER BY`를 그대로 쓰면서 여러 행을 더 적은 행으로 묶고, 그룹화 *이후*에 필터링하는 새 절(`HAVING`)을 추가합니다. 3장(`JOIN`)에서는 테이블에 별칭을 붙이는 데 `AS`를 끊임없이 재사용합니다. 그리고 예제 8의 `WHERE` + `ORDER BY` + `LIMIT` 조합은 이후 모든 챕터의 "자주 쓰는 조합" 섹션에서 거의 그대로 다시 등장합니다.

---
# 💼 Business Example
*How would a Business Analyst use this?*

**Scenario / 시나리오:**
**EN:** Your manager messages you on Slack: *"Can you pull the top 5 biggest completed orders this quarter? Just order ID, customer, and amount."* This is a two-minute query once you know WHERE + ORDER BY + LIMIT.
**KR:** 매니저가 슬랙으로 메시지를 보냅니다: *"이번 분기 완료된 주문 중 금액이 가장 큰 5건만 뽑아줄 수 있어? order ID, 고객, 금액만."* WHERE + ORDER BY + LIMIT을 알면 2분이면 끝나는 쿼리입니다.

**To-do / 할 일:**
- [x] Filter to only `완료` (completed) orders  
`완료` 상태인 주문만 필터링한다
- [x] Sort by `amount`, highest first  
`amount` 기준 내림차순 정렬한다
- [x] Keep only the top 5 rows  
상위 5건만 남긴다
- [x] Rename columns to something Slack-friendly with `AS`  
`AS`로 슬랙에 붙여넣기 좋은 열 이름으로 바꾼다

In [12]:
orders_biz = pd.DataFrame({
    "order_id":    [1001, 1002, 1003, 1004, 1005, 1006, 1007],
    "customer_id": ["C01","C02","C01","C03","C02","C04","C03"],
    "amount":      [45000, 320000, 61000, 28000, 950000, 15000, 410000],
    "status":      ["완료","완료","취소","완료","완료","완료","완료"],
})

sql = """
SELECT
    order_id  AS order_id,
    customer_id AS customer,
    amount    AS amount
FROM orders_biz
WHERE status = '완료'
ORDER BY amount DESC
LIMIT 5
"""
display(run(sql))


,order_id,customer,amount
0,1005,C02,950000
1,1007,C03,410000
2,1002,C02,320000
3,1001,C01,45000
4,1004,C03,28000


---
# 📝 Summary
*Write today's concept in 3~5 sentences.*

**EN:** `SELECT` and `FROM` pick the columns and table you want, and `WHERE`, `ORDER BY`, and `LIMIT` narrow, sort, and cap the rows you get back. `AND`/`OR`/`NOT` build compound filters — always parenthesize when mixing them. `AS` renames columns and tables in the output only, but it can't be reused inside `WHERE` because SQL actually runs `FROM → WHERE → SELECT → ORDER BY`, not top-to-bottom as written. `DISTINCT` removes duplicate rows, and simple arithmetic (`+ - * /`) lets you compute new columns like profit or margin directly in `SELECT`. Together, `WHERE` + `ORDER BY` + `LIMIT` form the single most common query shape in day-to-day BA work.

**KR:** `SELECT`와 `FROM`은 원하는 열과 테이블을 고르고, `WHERE`·`ORDER BY`·`LIMIT`은 돌려받는 행을 좁히고 정렬하고 개수를 제한합니다. `AND`/`OR`/`NOT`으로 복합 조건을 만들며, 섞어 쓸 때는 항상 괄호로 묶어야 합니다. `AS`는 결과에서만 열과 테이블 이름을 바꾸는데, SQL이 작성한 순서가 아니라 실제로는 `FROM → WHERE → SELECT → ORDER BY` 순으로 실행되기 때문에 `WHERE` 안에서는 재사용할 수 없습니다. `DISTINCT`는 중복 행을 제거하고, 간단한 산술 연산(`+ - * /`)은 `SELECT` 안에서 이익이나 마진 같은 새 열을 바로 계산하게 해줍니다. 이 모두를 합친 `WHERE` + `ORDER BY` + `LIMIT`는 BA의 일상 업무에서 가장 흔한 쿼리 형태입니다.

---
# 📌 One Sentence Summary
Today's topic in ONE sentence. / 오늘 배운 내용을 한 문장으로.

> **EN:** SQL runs `FROM → WHERE → SELECT → ORDER BY → LIMIT` behind the scenes, and once you internalize that order, WHERE/AND-OR-NOT/AS/ORDER BY/LIMIT/DISTINCT all click into place as one coherent system rather than separate rules to memorize.

> **KR:** SQL은 내부적으로 `FROM → WHERE → SELECT → ORDER BY → LIMIT` 순서로 실행되며, 이 순서를 이해하고 나면 WHERE/AND-OR-NOT/AS/ORDER BY/LIMIT/DISTINCT가 따로 외워야 할 규칙이 아니라 하나로 맞물린 시스템처럼 이해됩니다.

---
# ❓ Review Questions

**Q1.** What's the real execution order of `SELECT`, `FROM`, `WHERE`, and `ORDER BY` — and how does that explain why an alias works in `ORDER BY` but not in `WHERE`?
**Q1.** `SELECT`, `FROM`, `WHERE`, `ORDER BY`의 실제 실행 순서는 무엇이며, 이것이 왜 별칭이 `ORDER BY`에서는 되고 `WHERE`에서는 안 되는지를 어떻게 설명하는가?

**Q2.** Why does `WHERE dept = '개발' AND salary >= 5000000 OR dept = '마케팅'` need parentheses to reliably mean "(dev AND salary≥5M) OR marketing"?
**Q2.** 왜 이 조건은 "(개발팀이면서 연봉 5백 이상) 또는 (마케팅팀)"이라는 의미를 확실히 하려면 괄호가 필요한가?

**Q3.** You wrote `SELECT price - cost AS profit FROM products WHERE profit > 10000` and got an error. What's the fix, and why does it work?
**Q3.** `SELECT price - cost AS profit FROM products WHERE profit > 10000`를 실행했더니 오류가 났다. 해결책은 무엇이고 왜 그것이 통하는가?

**Q4.** Why is `LIMIT 5` without an `ORDER BY` considered risky in a real report?
**Q4.** 실무 리포트에서 `ORDER BY` 없는 `LIMIT 5`가 왜 위험하다고 여겨지는가?

**Q5.** For `SELECT DISTINCT customer_id, region FROM orders`, what exactly has to match for two rows to count as "duplicates" of each other?
**Q5.** `SELECT DISTINCT customer_id, region FROM orders`에서, 두 행이 서로 "중복"으로 취급되려면 정확히 무엇이 같아야 하는가?

---
*📅 Try answering these again in a few days. / 며칠 후 다시 답해보세요.*